# HLS Downloader
Downloads obfuscated HLS streams (TikTok CDN `.image`, `.woff2`, etc.) and saves to Google Drive.

In [ ]:
# ── CELL 1: Setup ────────────────────────────────────────────────────────────
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q requests tqdm ipywidgets

from google.colab import output as _colab_output
_colab_output.enable_custom_widget_manager()   # required for ipywidgets in Colab

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted. Widgets enabled.')

In [ ]:
# ── CELL 2: Config UI ────────────────────────────────────────────────────────
import ipywidgets as widgets, os, json
from IPython.display import display, clear_output
from google.colab import files as _colab_files

MOVIES       = []
DRIVE_FOLDER = 'Movies'
WORKERS      = 15
DRIVE_PATH   = f'/content/drive/MyDrive/{DRIVE_FOLDER}'

_queue = []   # list of {name, url, segments, stream_type}

# ── Widgets ───────────────────────────────────────────────────────────────────
w_folder  = widgets.Text(value='Movies', description='Drive folder:',
                          layout=widgets.Layout(width='260px'))
w_workers = widgets.IntSlider(value=15, min=1, max=30, step=1, description='Workers:',
                               style={'description_width': '70px'},
                               layout=widgets.Layout(width='300px'))
w_name    = widgets.Text(placeholder='Movie name  (e.g. Toy Story 5)',
                          layout=widgets.Layout(width='200px'))
w_url     = widgets.Text(placeholder='https://…/master.m3u8  — or leave blank if loading from JSON',
                          layout=widgets.Layout(width='460px'))
w_add     = widgets.Button(description='+ Add URL', button_style='primary',
                            layout=widgets.Layout(width='100px'))
w_upload  = widgets.Button(description='⬆ Load JSON', button_style='info',
                            layout=widgets.Layout(width='110px'))
w_clr     = widgets.Button(description='Clear all', button_style='warning',
                            layout=widgets.Layout(width='90px'))
w_go      = widgets.Button(description='✓ Apply', button_style='success',
                            layout=widgets.Layout(width='100px'))

out_queue  = widgets.Output()
out_status = widgets.Output()

def _refresh():
    with out_queue:
        clear_output(wait=True)
        if not _queue:
            print('  (empty)')
            return
        for i, e in enumerate(_queue, 1):
            kind = '[MP4]' if e.get('stream_type') == 'direct' else '[HLS]'
            if e['segments']:
                src = f"{len(e['segments'])} segs (pre-fetched)"
            else:
                src = e['url'][:60]
            print(f"  {i:>2}.  {kind}  {e['name']:<28}  {src}")

def on_add(_):
    n, u = w_name.value.strip(), w_url.value.strip()
    if not n or not u:
        with out_status: clear_output(wait=True); print('⚠  Fill both name and URL')
        return
    _queue.append({'name': n, 'url': u, 'segments': None, 'stream_type': 'hls'})
    w_name.value = w_url.value = ''
    _refresh()

def on_upload(_):
    """Upload the JSON exported by the M3U8 Detector extension (Save JSON button)."""
    with out_status: clear_output(wait=True); print('Select the m3u8_*.json file exported from the extension…')
    uploaded = _colab_files.upload()
    for fname, data in uploaded.items():
        try:
            entries = json.loads(data.decode())
        except Exception as ex:
            with out_status: clear_output(wait=True); print(f'⚠  Invalid JSON in {fname}: {ex}')
            return
        if not isinstance(entries, list):
            with out_status: clear_output(wait=True); print('⚠  JSON must be an array of stream entries')
            return
        added = 0
        for e in entries:
            segs        = e.get('segments') or []
            url         = e.get('streamUrl', '')
            stream_type = e.get('streamType', 'hls')
            name = (
                e.get('name') or
                e.get('customName') or
                e.get('pageTitle') or
                url.split('/')[-1].split('?')[0] or
                f'stream_{len(_queue)+1}'
            ).strip()
            if any(x['name'] == name for x in _queue):
                name = f"{name} ({len(_queue)+1})"
            if segs:
                _queue.append({'name': name, 'url': url, 'segments': segs, 'stream_type': stream_type})
            elif url:
                _queue.append({'name': name, 'url': url, 'segments': None, 'stream_type': stream_type})
            else:
                continue
            added += 1
        with out_status:
            clear_output(wait=True)
            print(f'✓ Loaded {added} stream(s) from {fname}')
            for x in _queue[-added:]:
                kind = '[MP4]' if x.get('stream_type') == 'direct' else '[HLS]'
                src  = f"{len(x['segments'])} segs pre-fetched" if x['segments'] else f"URL: {x['url'][:50]}"
                print(f"   • {kind} {x['name']}  →  {src}")
        _refresh()

def on_clear(_):
    _queue.clear()
    _refresh()

def on_apply(_):
    global MOVIES, DRIVE_FOLDER, WORKERS, DRIVE_PATH
    MOVIES       = [(e['name'], e['url'], e['segments'], e.get('stream_type', 'hls')) for e in _queue]
    DRIVE_FOLDER = w_folder.value.strip() or 'Movies'
    WORKERS      = w_workers.value
    DRIVE_PATH   = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    with out_status:
        clear_output(wait=True)
        print(f'✓ folder: MyDrive/{DRIVE_FOLDER} | workers: {WORKERS} | queued: {len(MOVIES)}')
        for n, u, segs, st in MOVIES:
            kind = '[MP4]' if st == 'direct' else '[HLS]'
            src  = f'{len(segs)} segs pre-fetched' if segs else u[:55]
            print(f'   • {kind} {n}  →  {src}')

w_add.on_click(on_add)
w_upload.on_click(on_upload)
w_clr.on_click(on_clear)
w_go.on_click(on_apply)

display(widgets.VBox([
    widgets.HTML('<b style="font-size:13px">── Settings ──</b>'),
    widgets.HBox([w_folder, w_workers]),
    widgets.HTML('<b style="font-size:13px">── Queue ──</b>'),
    widgets.HTML('<span style="font-size:11px;color:#888">Add by URL  —or—  load the JSON exported by the M3U8 Detector extension (auto-populates all streams)</span>'),
    widgets.HBox([w_name, w_url, w_add, w_upload, w_clr]),
    out_queue,
    widgets.HBox([w_go, out_status]),
]))
_refresh()

In [ ]:
# ── CELL 3: Functions ────────────────────────────────────────────────────────
import os, glob, shutil, subprocess, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from urllib.parse import urlparse

IEND      = b'IEND\xaeB\x60\x82'
PNG_MAGIC = b'\x89PNG'

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
})

def _origin(url):
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"

def fetch_text(url):
    r = SESSION.get(url, timeout=30, headers={"Referer": _origin(url)})
    r.raise_for_status()
    return r.text

def get_segments(playlist_url):
    """Returns list of segment URLs. Resolves master playlist → highest quality."""
    content = fetch_text(playlist_url)
    if '#EXT-X-STREAM-INF' in content:
        lines = content.strip().splitlines()
        variants = []
        for i, line in enumerate(lines):
            if line.startswith('#EXT-X-STREAM-INF'):
                bw = next((int(p.split('=')[1]) for p in line.split(',') if p.startswith('BANDWIDTH=')), 0)
                uri = lines[i+1].strip()
                if not uri.startswith('http'):
                    uri = playlist_url.rsplit('/', 1)[0] + '/' + uri
                variants.append((bw, uri))
        variants.sort(reverse=True)
        print(f"  Variants: {[f'{bw//1000}k' for bw,_ in variants]} → selecting {variants[0][0]//1000}k")
        playlist_url = variants[0][1]
        content = fetch_text(playlist_url)
    segs = []
    for line in content.strip().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            if not line.startswith('http'):
                line = playlist_url.rsplit('/', 1)[0] + '/' + line
            segs.append(line)
    return segs

def download_direct(url, output_path):
    """Downloads a direct video file (no segments) to output_path with progress bar."""
    r = SESSION.get(url, stream=True, timeout=60, headers={"Referer": _origin(url)})
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(output_path, 'wb') as f, tqdm(
        total=total or None, unit='B', unit_scale=True, desc='  Downloading'
    ) as bar:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
            bar.update(len(chunk))
    return os.path.getsize(output_path) / 1024 / 1024

def download_all(segments, workdir, workers):
    """Downloads segments in parallel. Returns list of failed indices."""
    def _dl(args):
        idx, url = args
        out = f"{workdir}/seg-{idx:04d}.ts"
        if os.path.exists(out):
            return idx, True
        try:
            r = SESSION.get(url, timeout=30, headers={"Referer": _origin(url)})
            r.raise_for_status()
            with open(out, 'wb') as f: f.write(r.content)
            return idx, True
        except:
            return idx, False

    tasks = list(enumerate(segments, 1))
    failed = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(_dl, t): t for t in tasks}
        with tqdm(total=len(tasks), unit='seg', desc='  Downloading') as bar:
            for f in as_completed(futures):
                idx, ok = f.result()
                if not ok: failed.append(idx)
                bar.update(1)
    return failed

def retry_failed(failed, segments, workdir):
    """Retries failed segments one by one with a longer timeout."""
    still = []
    for idx in tqdm(failed, desc='  Retrying', unit='seg'):
        out = f"{workdir}/seg-{idx:04d}.ts"
        try:
            url = segments[idx-1]
            r = SESSION.get(url, timeout=60, headers={"Referer": _origin(url)})
            r.raise_for_status()
            with open(out, 'wb') as f: f.write(r.content)
        except:
            still.append(idx)
    return still

def combine(workdir):
    """Detects segment type, strips PNG wrapper if needed, combines to combined.ts."""
    segs = sorted(glob.glob(f"{workdir}/seg-*.ts"),
                  key=lambda p: int(os.path.basename(p).split('seg-')[1].split('.')[0]))
    sample = open(segs[0], 'rb').read(4)
    has_wrapper = sample == PNG_MAGIC
    print(f"  Segment type: {'PNG wrapper (TikTok CDN)' if has_wrapper else 'raw MPEG-TS'}")
    combined = f"{workdir}/combined.ts"
    with open(combined, 'wb') as out:
        for path in tqdm(segs, unit='seg', desc='  Combining'):
            data = open(path, 'rb').read()
            if has_wrapper:
                pos = data.find(IEND)
                if pos != -1: data = data[pos + len(IEND):]
            out.write(data)
    return combined

def mux(combined, output_path):
    """Muxes combined.ts → MP4. Returns (size_mb, duration_str)."""
    result = subprocess.run(
        ['ffmpeg', '-y', '-i', combined, '-c', 'copy', '-bsf:a', 'aac_adtstoasc', output_path],
        capture_output=True, text=True
    )
    if not os.path.exists(output_path):
        raise RuntimeError(result.stderr[-1000:])
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    probe = subprocess.run(['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
                            '-of', 'csv=p=0', output_path], capture_output=True, text=True)
    try:
        s = float(probe.stdout.strip())
        dur = f"{int(s//3600)}h{int((s%3600)//60)}m{int(s%60)}s"
    except:
        dur = '?'
    return size_mb, dur

def probe_duration(path):
    probe = subprocess.run(['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
                            '-of', 'csv=p=0', path], capture_output=True, text=True)
    try:
        s = float(probe.stdout.strip())
        return f"{int(s//3600)}h{int((s%3600)//60)}m{int(s%60)}s"
    except:
        return '?'

print('Functions loaded. UA set ✓')

In [ ]:
# ── CELL 4: Download all movies ──────────────────────────────────────────────
results = []

for i, (name, url, prefetched_segments, stream_type) in enumerate(MOVIES, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(MOVIES)}] {name}  ({'Direct MP4' if stream_type == 'direct' else 'HLS'})")
    print(f"{'='*60}")

    safe_name   = name.replace(' ', '_')
    workdir     = f"/content/hls_{i:02d}_{safe_name}"
    output_path = f"{DRIVE_PATH}/{name}.mp4"

    if os.path.exists(output_path):
        print(f"  Already exists → skipping")
        results.append((name, 'skipped', '-', '-'))
        continue

    try:
        # ── Direct MP4 (mp4upload, etc.) ─────────────────────────────────────
        if stream_type == 'direct':
            print(f"  Source: {url[:70]}...")
            size_mb = download_direct(url, output_path)
            dur = probe_duration(output_path)
            print(f"  ✓ {name}.mp4 — {size_mb:.0f} MB, {dur}")
            results.append((name, 'ok', f"{size_mb:.0f} MB", dur))
            continue

        # ── HLS ──────────────────────────────────────────────────────────────
        os.makedirs(workdir, exist_ok=True)

        # 1. Get segment list
        if prefetched_segments:
            segments = prefetched_segments
            print(f"  Segments: {len(segments)} (pre-fetched from extension)")
        else:
            print(f"  Playlist: {url[:70]}...")
            segments = get_segments(url)
            print(f"  Segments: {len(segments)}")

        # 2. Parallel download
        failed = download_all(segments, workdir, WORKERS)
        print(f"  Downloaded: {len(segments)-len(failed)}/{len(segments)}")

        # 3. Retry failures
        if failed:
            print(f"  Retrying {len(failed)} failed segments...")
            still = retry_failed(failed, segments, workdir)
            if still:
                print(f"  Still failed (skipping): {still}")

        # 4. Combine + strip PNG wrapper if present
        combined = combine(workdir)

        # 5. Mux → Drive
        print(f"  Muxing → {output_path}")
        size_mb, dur = mux(combined, output_path)
        print(f"  ✓ {name}.mp4 — {size_mb:.0f} MB, {dur}")
        results.append((name, 'ok', f"{size_mb:.0f} MB", dur))

        shutil.rmtree(workdir)

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append((name, 'error', str(e)[:60], '-'))

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for name, status, size, dur in results:
    icon = '✓' if status == 'ok' else ('⚠' if status == 'skipped' else '✗')
    print(f"  {icon} {name:30s}  {status:8s}  {size:8s}  {dur}")